# 01. QuartzNet 10x4 — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: QuartzNet-10x4 (TCS-свёрточный энкодер, ~4M параметров).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [5]:
import gdown
gdown.download(url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link", output="/content/data.zip", quiet=False, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW
From (redirected): https://drive.google.com/uc?id=1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW&confirm=t&uuid=0608aa38-fec5-483d-91bd-5e4531201716
To: /content/data.zip
100%|██████████| 1.77G/1.77G [00:16<00:00, 109MB/s]


'/content/data.zip'

In [6]:
import zipfile
zipfile.ZipFile("/content/data.zip").extractall("data")

## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [1]:
from pathlib import Path
import torch

# Fill in before running.
TRAIN_ROOT = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/train")       # dir with train audio files (.wav / .mp3)
DEV_ROOT = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/dev")         # dir with dev audio files
TEST_ROOT: Path | None = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/test")        # optional, set to Path("...") if have test data
TRAIN_CSV = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/train/train.csv")        # Kaggle-style CSV: filename, transcription, spk_id, gender, ext, samplerate
DEV_CSV = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/dev/dev.csv")
CKPT_ROOT = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/checkpoints")        # where best.pt + meta.json will be saved

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

device=cuda, train=/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/train, dev=/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/dev, ckpts=/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/checkpoints


## Шаг 1. Импорты

In [2]:
import json
import random
import time
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
# Должно быть выставлено до import torch (точнее до первой CUDA-аллокации).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ и Blackwell без потери CER.
torch.set_float32_matmul_precision("high")

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [3]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 60,
    "grad_accum": 2,
    "subsample_factor": 2,
}
HP_GRID = {
    "lr":                        [0.010, 0.015, 0.020],
    "weight_decay":              [1e-3],
    "dropout":                   [0.10, 0.15, 0.20],
    "d_model":                   [256],
    "warmup_steps":              [200, 500, 1000],
    "specaug_freq_mask_param":   [15, 20],
    "specaug_time_mask_param":   [20, 25],
    "speed_prob":                [1.0],
    "pitch_prob":                [0.3],   # speed XOR pitch — обе не сработают одновременно
    "gain_prob":                 [0.3, 0.7],
    "noise_prob":                [0.0],   # требует MUSAN корпуса (musan_root)
}
N_TRIALS = 8
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)

FIXED: {'samplerate': 16000, 'n_fft': 512, 'n_mels': 80, 'hop_length': 160, 'win_length': 400, 'max_epochs': 60, 'grad_accum': 2, 'subsample_factor': 2}
N_TRIALS: 8


## Шаг 3. Загрузка записей из CSV

In [4]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")

02:49:39 records_from_csv: loaded 12553 records from /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/train/train.csv
02:49:39 records_from_csv: loaded 2265 records from /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/dev/dev.csv
Train records: 12553, Dev records: 2265


## Шаг 4. Vocabulary

In [5]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

Vocab size: 35, blank_id: 0


In [6]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()

preload audio: 100%|██████████| 14818/14818 [00:14<00:00, 1029.73it/s]

02:49:54 preload_audio_cache: 14818 tensors, 2.96 GB RAM


## Шаг 5. HP Random Search — Training Loop

In [ ]:
import gc
import math

SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"01_quartznet_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
DL_WORKERS = 4  # speed/pitch perturb на CPU — pipeline нужны воркеры под GPU


def _worker_init(worker_id: int) -> None:
    """1 BLAS-тред на воркер + индивидуальный сид RNG аугментера.

    Без пересева у каждого воркера _augmenter._rng стартует с одного
    base_seed → одинаковые последовательности аугментаций (на разных
    сэмплах). PyTorch уже задаёт info.seed детерминированно от
    base_seed загрузчика и worker_id — используем его.
    """
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        gain_prob=hp["gain_prob"],
        noise_prob=hp["noise_prob"],
        vtlp_prob=0.0,
        rir_prob=0.0,
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=2,
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=5,
        time_mask_max_ratio=0.05,
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        worker_init_fn=_worker_init,
    )

    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)
    model = torch.compile(model, mode="default", dynamic=True)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_novograd(
        model.parameters(),
        lr=hp["lr"],
        betas=(0.95, 0.5),
        weight_decay=hp["weight_decay"],
    )
    # math.ceil учитывает «хвостовой» optimizer.step(), который Trainer
    # делает при len(loader) % grad_accum != 0 — иначе scheduler уходит
    # в min_lr_ratio за несколько эпох до конца.
    steps_per_epoch = math.ceil(len(train_loader) / FIXED["grad_accum"])
    total_steps = steps_per_epoch * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=0.01,
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        amp_dtype=torch.bfloat16,
        val_every_n_epochs=1,
        early_stop_patience=15,
        early_stop_metric="max_speaker_cer",
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "quartznet_10x4",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_val_cer"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

    # Cleanup в конце trial.
    del trainer, model, optimizer, scheduler, ctc
    del train_loader, dev_loader, train_ds, dev_ds
    del train_aug, spec_aug
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    torch._dynamo.reset()

print("\nHP search complete.")



=== Trial 1/8 ===
{
  "samplerate": 16000,
  "n_fft": 512,
  "n_mels": 80,
  "hop_length": 160,
  "win_length": 400,
  "max_epochs": 60,
  "grad_accum": 2,
  "subsample_factor": 2,
  "lr": 0.02,
  "weight_decay": 0.001,
  "dropout": 0.1,
  "d_model": 256,
  "warmup_steps": 200,
  "specaug_freq_mask_param": 15,
  "specaug_time_mask_param": 20,
  "speed_prob": 1.0,
  "pitch_prob": 0.3,
  "gain_prob": 0.7,
  "noise_prob": 0.0
}
02:49:55 QuartzNet10x4 initialised: 4352371 params (target ~4.0M)


epochs:   0%|          | 0/60 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/197 [00:00<?, ?it/s]

02:50:13 epoch 1 step 50 loss=7.0276
02:50:24 validation epoch 1 corpus_cer=0.9761
02:50:24 epoch 1 val_cer=0.9761 max_spk=0.9964 best=0.9964 no_improve=0/15


epoch 2:   0%|          | 0/197 [00:00<?, ?it/s]

...

05:23:36 epoch 58 step 5650 loss=0.3290
05:23:44 epoch 58 step 5700 loss=0.2793
05:23:54 validation epoch 58 corpus_cer=0.1369
05:23:54 epoch 58 val_cer=0.1369 max_spk=0.2128 best=0.2118 no_improve=3/15


epoch 59:   0%|          | 0/197 [00:00<?, ?it/s]

05:23:55 epoch 59 step 5750 loss=0.2613
05:24:03 epoch 59 step 5800 loss=0.2550
05:24:13 validation epoch 59 corpus_cer=0.1383
05:24:13 epoch 59 val_cer=0.1383 max_spk=0.2171 best=0.2118 no_improve=4/15


epoch 60:   0%|          | 0/197 [00:00<?, ?it/s]

05:24:14 epoch 60 step 5850 loss=0.3055
05:24:23 epoch 60 step 5900 loss=0.3096
05:24:32 validation epoch 60 corpus_cer=0.1379
05:24:32 epoch 60 val_cer=0.1379 max_spk=0.2186 best=0.2118 no_improve=5/15
trial 7: peak VRAM = 4.88 GB

HP search complete.


## Шаг 6. Сводный отчёт трайлов

In [8]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_val_cer"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'d_model':>8}")
print("-" * 50)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_val_cer']:>12.4f}"
        f" {hp['lr']:>8.4f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
    )

trial best_val_cer       lr  dropout  d_model
--------------------------------------------------
    0       0.1839   0.0200   0.1000      256
    7       0.2118   0.0200   0.1000      256
    3       0.3098   0.0150   0.1500      256
    4       0.3109   0.0200   0.2000      256
    6       0.3290   0.0200   0.2000      256
    1       0.3324   0.0100   0.1000      256
    5       0.4552   0.0100   0.1500      256
    2       0.5560   0.0100   0.2000      256


## Шаг 7. Beam search + KenLM rescore

In [13]:
from pathlib import Path
from gp1.lm.build_corpus import build_synthetic_corpus

LM_DIR = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/lm")
corpus_path = LM_DIR / "corpus.txt"

n_lines = build_synthetic_corpus(
    out_path=corpus_path,
    train_manifest=None,   # опционально: путь к JSONL с полем "transcription"
)
print(f"Corpus: {n_lines} lines → {corpus_path}")

15:24:41 Building synthetic corpus → /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/lm/corpus.txt
15:24:47 Synthetic corpus: 999999 unique word forms
15:24:48 Corpus written: 999999 lines → /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/lm/corpus.txt
Corpus: 999999 lines → /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/lm/corpus.txt


## Шаг 8. Submission (если test данные доступны)

In [ ]:
if TEST_ROOT is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # Pair filenames (preserved order from SequentialSampler) with predictions.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
